# 1 · Raster fields

The raster builders turn a pyramids `Dataset` into an interactive field: `field` (regular grid),
`quadmesh` (irregular/curvilinear cells), `contours` (iso-lines, filled or not), `rgb`
(three-band composite) and `spaghetti` (one contour set per ensemble member). NoData is rendered
transparent automatically.

**Setup** — load the backends and the bundled Lisbon DEM (a clean single-band raster, EPSG:4326).

In [ ]:
import warnings
warnings.filterwarnings("ignore")            # hide backend-mismatch style warnings in the docs

from pathlib import Path

# Resolve the repo root so the bundled sample data is found whether this runs from
# docs/examples/interactive/ (mkdocs) or the repository root.
ROOT = Path.cwd()
while not (ROOT / "examples" / "data" / "LisbonElevation.tif").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DATA = ROOT / "examples" / "data"

import io
from IPython.display import Image
import matplotlib.pyplot as plt
import holoviews as hv
hv.extension("bokeh", "matplotlib")          # Bokeh = interactive; matplotlib = static (projection/barbs)

def show(m, static=False, dpi=96):
    """Display the map — **interactive Bokeh** by default (pan / zoom / hover).

    RUN THIS NOTEBOOK (Run All / Shift-Enter) to see the live interactive maps: a saved .ipynb only
    shows interactive plots once the viewer executes their JavaScript (a running kernel, or a trusted
    notebook). Pass ``static=True`` (used for the matplotlib-only layers — projection, streamlines,
    wind barbs, which have no Bokeh glyph) to render a static PNG instead.
    """
    if static or getattr(m, "_projection", None) is not None:
        fig = hv.render(m.render(), backend="matplotlib")
        buf = io.BytesIO()
        fig.savefig(buf, format="png", dpi=dpi, bbox_inches="tight")
        plt.close(fig)
        return Image(data=buf.getvalue())
    return m.render()                        # interactive Bokeh object

from pyramids.dataset import Dataset, GeoReference
from digitalearth.interactive import InteractiveMap

dem = Dataset.read_file(str(DATA / "LisbonElevation.tif"))

### `field` — a pixel field
`field` draws the raster as a regular pixel grid. `cmap=None` (the default) picks a colormap from the variable via auto-style; here we ask for `terrain`.

In [ ]:
m = InteractiveMap(crs=dem.epsg, title="image")
m.field(dem, cmap="terrain")
show(m)

### `quadmesh` — irregular cells
`quadmesh` draws every cell from its coordinate arrays, so non-uniform or curvilinear grids render without resampling. On a regular grid it matches `field`.

In [ ]:
m = InteractiveMap(crs=dem.epsg, title="quadmesh")
m.quadmesh(dem, cmap="viridis")
show(m)

### `contours` — iso-lines
`contours` draws line iso-contours, and `filled=True` fills the bands between them instead. Either takes a
`levels` count or an `interval` spacing.

In [ ]:
m = InteractiveMap(crs=dem.epsg, title="contours (filled)")
m.contours(dem, levels=10, filled=True)
show(m)

### `rgb` — a three-band composite
A false-colour seasonal composite of the global temperature stack: **R = January, G = July, B = December**. The ocean NoData sentinel is masked to `NaN` first so the per-band 2–98 % stretch sees only real temperatures (green = boreal summer, magenta = austral/tropical warmth, ocean transparent).

In [ ]:
import numpy as np

# The WorldClim global stack lives in the gitignored examples/data/global/, so render this only when the sample
# is present locally; in CI / a fresh clone it is absent and the cell skips with a note.
_tavg = DATA / "global" / "wc2.1_10m_tavg_01.tif"
if _tavg.exists():
    def _temp(month):
        a = Dataset.read_file(str(DATA / "global" / f"wc2.1_10m_tavg_{month:02d}.tif")).read_array(band=0)
        return np.where(a.astype("float32") < -1e30, np.nan, a)   # ocean sentinel -> transparent

    stack = np.stack([_temp(1), _temp(7), _temp(12)]).astype("float32")
    _src = Dataset.read_file(str(_tavg))
    composite = Dataset.from_array(
        arr=stack,
        geo_ref=GeoReference(geo=_src.geotransform, epsg=_src.epsg),
        no_data_value=np.nan,
    )

    m = InteractiveMap(crs=4326, title="seasonal RGB (Jan/Jul/Dec)")
    m.rgb(composite)
    display(show(m))
else:
    print("skipped — the WorldClim sample (examples/data/global/) is not bundled with the repo")

### `spaghetti` — an ensemble of contours
Given a `DatasetCollection`, `spaghetti` overlays one contour set per member in a colour cycle — the classic 'spaghetti plot' for comparing members. Here the first four months of the temperature stack.

In [ ]:
from pyramids.dataset.collection import DatasetCollection

# WorldClim stack (gitignored) — render locally when present, skip with a note in CI / a fresh clone.
_tavg = DATA / "global" / "wc2.1_10m_tavg_01.tif"
if _tavg.exists():
    months = [str(DATA / "global" / f"wc2.1_10m_tavg_{m:02d}.tif") for m in range(1, 5)]
    ens = DatasetCollection.from_files(months)

    m = InteractiveMap(crs=4326, title="spaghetti (4 months)")
    m.spaghetti(ens, levels=4)
    display(show(m))
else:
    print("skipped — the WorldClim sample (examples/data/global/) is not bundled with the repo")